# Multi-Linear Regression Model of Student Assessment Performance Questionnaires
**by James Abello, Haoyang Zhang**

```mermaid
graph TB

subgraph Baseline_Model
    Build(Build a Multiple Linear Regression Model)
    Evaluate(Evaluate Model Performance using Adjusted R-Squared)
end

subgraph Feature_Selection
    Removal(Iterative Feature Removal and Adjusted R-Squared Calculation)
    Rank(Rank Features by Importance)
    Selection(Final Model Selection by Taking the Highest Adjusted R-Squared Model)
end

Build --> Evaluate
Evaluate --> Removal
Removal --> Rank
Rank --> Selection
Rank --> Removal
```

## Dataset Description: Student Assessment Questionnaires and Quiz Scores

The dataset `assessment_quiz_generated.csv` contains information derived from student assessment questionnaires and their corresponding quiz scores.

The dataset comprises the following attributes:

- **`timestamp`**  
  The date and time when the assessment was submitted, formatted as `yyyy-mm-dd hh:mm:ss timezone`.

- **`netid`**  
  The encoded NetID of the student. Valid NetIDs must have a string length between 8 and 14 characters (inclusive). Entries falling outside this range are considered invalid.

- **`ruid`**  
  The encoded RUID of the student. A valid RUID is expected to contain exactly 18 characters. Any deviation from this length is considered invalid.

- **Skill Proficiency Columns**  
  The following columns record students’ self-assessed proficiency levels in specific skills, rated on scales ranging from 0 up to a multiple of 5 (depending on the number of questions per topic).

  - 0: `data_structures`  
  - 1: `calculus_and_linear_algebra`  
  - 2: `probability_and_statistics`  
  - 3: `data_visualization`  
  - 4: `python_libraries`  
  - 5: `shell_scripting`  
  - 6: `sql`  
  - 7: `python_scripting`  
  - 8: `jupyter_notebook`  
  - 9: `regression`  
  - 10: `programming_languages`  
  - 11: `algorithms`  
  - 12: `complexity_measures`  
  - 13: `visualization_tools`  
  - 14: `massive_data_processing`

- **`quiz_score` (New Added column to the previous dataset)**
  The score obtained by the student in the quiz, represented as a floating-point number between 0 and 100 (inclusive).

## **Assessment Tasks**

Complete the following tasks by using the provided Jupyter Notebook Template `ME-AR.ipynb` in the [folder "Model Evaluation"](https://rutgersconnect-my.sharepoint.com/:f:/g/personal/hz333_connect_rutgers_edu/EjQo7aXXTqJJu79rqrgiJPgBli8zYMXHfoPYzvgiPaah3A).

### Part 1: Baseline Model Construction
1.  **Build a Multiple Linear Regression Model** (Task 1.1): Construct a multiple linear regression model using `train_linear_regression_model` to predict `quiz_score` using all 15 skill proficiency columns as predictor variables.
2.  **Evaluate Model Performance using Adjusted R-Squared** (Task 1.2): Evaluate the performance of the baseline model by calculating and reporting its adjusted R-squared value. The formula for adjusted R-squared is:
    $$
    \text{Adjusted } R^2 = 1 - (1 - R^2) \frac{n - 1}{n - p - 1}
    $$
    where $n$ is the number of observations and $p$ is the number of predictors.

### Part 2: Feature Importance Analysis
1.  **Iterative Feature Removal and Adjusted R-Squared Calculation** (Task 2.1): Systematically evaluate the importance of each feature by performing the following steps for each of the 15 skill proficiency columns:
    - a. Temporarily remove one feature from the dataset.
    - b. Run the linear regression model on the remaining 14 features.
    - c. Calculate the adjusted R-squared of the 14-features model.
    - d. Report the difference in adjusted R-squared compared to the adjusted R-squared of the baseline 15-features model.
2.  **Rank Features by Importance** (Task 2.2): Rank the 15 skill proficiency features from most to least important based on the magnitude of the drop in adjusted R-squared when the feature was removed. A larger drop indicates greater importance.
3.  **Final Model Selection by Taking the Highest Adjusted R-Squared Model** (Task 2.3): Based on the importance of the features, decide the number and the subset of skill proficiency features to be used in the Final Multi-Linear Regression Model.
4.  **Plot number of features used vs. the Adjusted R-Squared** (Task 2.4): Draw a line plot, the x-axis is the number of features used, and the y-axis is the Adjusted R-Squared value.

### Environment Setup and Data Loading

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
def load_data(file_path):
    """Load the dataset from a CSV file.
    IN: file_path: str, path to the CSV file
    OUT: pd.DataFrame, loaded dataset
    """
    return pd.read_csv(file_path)

### Task 1.1: Build a Muti-Linear Regression Model

In [ ]:
def train_linear_regression_model(df, feature_cols, target_col):
    """Train a linear regression model.
    IN: df: pd.DataFrame, the dataset
        feature_cols: list of str, names of feature columns
        target_col: str, name of the target column
    OUT: LinearRegression, trained model
    """
    X = df[feature_cols]
    y = df[target_col]
    model = LinearRegression()
    model.fit(X, y)

    return model

### Task 1.2: Evaluate Model Performance using Adjusted R-Squared

In [ ]:
def calculate_adjusted_r_squared(model, df, feature_cols, target_col):
    """Calculate the adjusted R-squared of the model.
    IN: model: LinearRegression, trained model
        df: pd.DataFrame, the dataset
        feature_cols: list of str, names of feature columns
        target_col: str, name of the target column
    OUT: float, adjusted R-squared value
    """
    X = df[feature_cols]
    y = df[target_col]
    n = len(df)
    p = len(feature_cols)
    r_squared = model.score(X, y)
    adjusted_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - p - 1)

    return adjusted_r_squared

### Task 2: Feature Importance Analysis

#### 2.1. Iterative Feature Removal and Adjusted R-Squared Calculation

In [ ]:
def calculate_reduced_model_adjusted_r_squared(df, feature_cols, target_col, remove_col):
    """Calculate adjusted R-squared after removing one feature.
    IN: df: pd.DataFrame, the dataset
        feature_cols: list of str, names of current feature columns
        target_col: str, name of the target column
        remove_col: str, name of the feature to remove
    OUT: float, adjusted R-squared value of the reduced model
    """
    reduced_cols = [col for col in feature_cols if col != remove_col]
    model = train_linear_regression_model(df, reduced_cols, target_col)
    adjusted_r2 = calculate_adjusted_r_squared(model, df, reduced_cols, target_col)
    return adjusted_r2


#### 2.2. Rank Features by Importance

In [ ]:
def reduce_model(df, feature_cols, target_col):
    """Remove the least important feature based on adjusted R-squared.
    IN: df: pd.DataFrame, the dataset
        feature_cols: list of str, names of current feature columns
        target_col: str, name of the target column
    OUT: remaining_cols, list of str, updated feature columns after removal
         removed_col, str, name of the removed feature
         adjusted_r2, float, adjusted R-squared of the reduced model
    """
    best_adj_r2 = float('-inf')
    removed_col = None

    for col in feature_cols:
        adj_r2 = calculate_reduced_model_adjusted_r_squared(df, feature_cols, target_col, col)
        if adj_r2 > best_adj_r2:
            best_adj_r2 = adj_r2
            removed_col = col

    remaining_cols = [col for col in feature_cols if col != removed_col]
    return remaining_cols, removed_col, best_adj_r2

#### Subtask 2.3: Final Model Selection by Taking the Highest Adjusted R-Squared Model
Repeat the process of removing the least important feature (as identified in Subtask 2.2) until only one feature remains. After each removal, retrain the model and record the adjusted R-squared value.

In [ ]:
def feature_importance_ranking(df, feature_cols, target_col):
    """Iteratively remove the least important feature and rank features by importance.
    IN: df: pd.DataFrame, the dataset
        feature_cols: list of str, names of current feature columns
        target_col: str, name of the target column
    OUT: list of tuples (feature_name, adjusted_r_squared) in order of removal
    """
    ranking = []
    current_features = feature_cols.copy()

    full_model = train_linear_regression_model(df, current_features, target_col)
    full_adjusted_r2 = calculate_adjusted_r_squared(full_model, df, current_features, target_col)
    ranking.append(('Full Model', full_adjusted_r2))

    # Your code here
    while len(current_features) > 1:
        current_features, removed_col, adjusted_r2 = reduce_model(df, current_features, target_col)
        ranking.append((removed_col, adjusted_r2))

    return ranking

#### Subtask 2.4 Plot number of features used vs. the Adjusted R-Squared

In [ ]:
def plot_adjusted_r_2(ranking, skill_columns):
    """
    Draw a line plot of adjusted R-squared values against the number of features.

    This function plots the adjusted R-squared value for models with a decreasing number of features,
    from a full model down to a single-feature model. It also annotates each point with the index
    of the feature that was removed to achieve the next model, indicating the least important feature at each step.

    IN: ranking, list of tuples (feature_name, adjusted_r_squared) in order of removal
        skill_columns, list of str, names of all feature columns

    OUT:
    """
    plt.figure(figsize=(9, 6))

    # Your code here
    feature_counts = [len(skill_columns) - idx for idx in range(len(ranking))]
    adjusted_scores = [score for _, score in ranking]

    plt.plot(feature_counts, adjusted_scores, marker='o')
    plt.xticks(feature_counts)

    for idx, (removed_col, _) in enumerate(ranking[1:]):
        feature_index = skill_columns.index(removed_col)
        plt.annotate(str(feature_index),(feature_counts[idx], adjusted_scores[idx]),xytext=(0, 8),textcoords='offset points',ha='center')


    plt.xlabel('p = Number of Features Used')
    plt.ylabel('Adjusted R-Squared')
    plt.title('Model Performance vs. Number of Features')
    plt.grid(True)
    plt.gca().invert_xaxis()  # Invert x-axis to show features being removed from left to right
    plt.show()